In [ ]:
# -*- coding: utf-8 -*-
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or
# implied.
# See the License for the specific language governing permissions and
# limitations under the License.


# Imports

In [ ]:
from zenodo_get import download
from lxml import etree
import zipfile
import pandas as pd
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
import matplotlib.pyplot as plt
from sklearn.decomposition import TruncatedSVD
import os
from util.preprocessing import TweetPreprocessor 
import joblib

# Constant

In [ ]:
DATASETS_DIR = os.path.join(os.getcwd(), "datasets")
MODELS_DIR = os.path.join(os.getcwd(), "models")

# Dataset 100 tweets per author

Pan clef dataset from author profiling task

https://zenodo.org/records/3692340

https://doi.org/10.5281/zenodo.3692340

In [ ]:
download(record_or_doi="3692340", output_dir=DATASETS_DIR)

Remove not used parts

In [ ]:
unused_files_path = [
    "pan19-author-profiling-earlybirds-20190320.zip",
    "pan19-author-profiling-earlybirds-20190320.zip",
    "pan19-author-profiling-20200229.zip"
]

In [ ]:
for file in unused_files_path:
    file_path = os.path.join(DATASETS_DIR, file)
    if os.path.exists(file_path):
        if os.path.isfile(file_path):
            os.remove(file_path)
        else:
            os.rmdir(file_path)

In [ ]:
files_for_extract = [
    "pan19-author-profiling-training-dataset-2019-02-18.zip",
    "pan19-author-profiling-test-2019-04-29.zip"
]

In [ ]:
for file in files_for_extract:
    file_path = os.path.join(DATASETS_DIR, file)
    if os.path.exists(file_path) and zipfile.is_zipfile(file_path):
        with zipfile.ZipFile(file_path, 'r') as zip_ref:
            zip_ref.extractall(DATASETS_DIR)
        os.remove(file_path)

In [ ]:
training_path = os.path.join(DATASETS_DIR, "pan19-author-profiling-training-2019-02-18", "en")
if not os.path.exists(training_path):
    raise FileNotFoundError(f"Training path not found: {training_path}")
test_path = os.path.join(DATASETS_DIR, "pan19-author-profiling-test-2019-04-29", "en")
if not os.path.exists(test_path):
    raise FileNotFoundError(f"Test path not found: {test_path}")
test_truth_path = os.path.join(DATASETS_DIR, "pan19-author-profiling-test-2019-04-29", "en.txt")
if not os.path.exists(test_truth_path):
    raise FileNotFoundError(f"Test truth path not found: {test_truth_path}")

In [ ]:
rows = []

for file in os.listdir(training_path):
    if not file.endswith(".xml"):
        continue

    file_id = file.split(".")[0]
    xml_path = os.path.join(training_path, file)

    try:
        tree = etree.parse(xml_path)
        documents = tree.xpath("//document/text()")  # все твиты автора

        for doc in documents:
            rows.append({"document": doc, "id": file_id})

    except Exception as e:
        print(f"Error reading file {file}: {e}")

train_df = pd.DataFrame(rows)

truth_train_df = pd.read_csv(
    os.path.join(training_path, "truth.txt"),
    sep=":::",
    header=None,
    names=["id", "who", "gender_label"],
    engine="python",
)

train_df = train_df.merge(truth_train_df, on="id", how="left")
train_df.head()

In [ ]:
train_df.isna().sum()

In [ ]:
train_df = train_df[train_df["who"] != "bot"]
train_df["who"].value_counts()

In [ ]:
rows = []

for file in os.listdir(test_path):
    if not file.endswith(".xml"):
        continue

    file_id = file.split(".")[0]
    xml_path = os.path.join(test_path, file)

    try:
        tree = etree.parse(xml_path)
        documents = tree.xpath("//document/text()")

        for doc in documents:
            rows.append({"document": doc, "id": file_id})

    except Exception as e:
        print(f"Error reading file {file}: {e}")

test_df = pd.DataFrame(rows)

truth_test_df = pd.read_csv(
    test_truth_path,
    sep=":::",
    header=None,
    names=["id", "who", "gender_label"],
    engine="python",
)
test_df = test_df.merge(truth_test_df, on="id", how="left")
test_df.head()

In [ ]:
test_df.isna().sum()

delete bots

In [ ]:
test_df = test_df[test_df["who"] != "bot"]
test_df["who"].value_counts()

In [ ]:
if "document" not in train_df.columns:
	raise ValueError("No text column found in the dataset.")
text_col = "document"

if "id" not in train_df.columns:
	raise ValueError("No userid column found in the dataset.")
id_col = "id"

if "gender_label" not in train_df.columns:
	raise ValueError("No gender_label column found in the dataset.")
gender_label_col = "gender_label"

train_eval_df = train_df[[text_col, gender_label_col, id_col]].dropna().copy()
test_eval_df = test_df[[text_col, gender_label_col, id_col]].dropna().copy()

tweet_preprocessor = TweetPreprocessor(user_id_col=id_col, text_col=text_col)

train_eval_concat_df = tweet_preprocessor.concatenate(train_eval_df)
test_eval_concat_df = tweet_preprocessor.concatenate(test_eval_df)

x_train = train_eval_concat_df[text_col].astype(str)
y_train = train_eval_concat_df[gender_label_col].map({'female': 0, 'male': 1})
x_test = test_eval_concat_df[text_col].astype(str)
y_test = test_eval_concat_df[gender_label_col].map({'female': 0, 'male': 1})

In [ ]:
print("Train (gender_label)")
train_counts = train_df["gender_label"].value_counts()
train_percent = (train_df["gender_label"].value_counts(normalize=True) * 100).round(2)
print(f"count: {train_counts}, percent: {train_percent}")

print("Test (gender_label)")
test_counts = test_df["gender_label"].value_counts()
test_percent = (test_df["gender_label"].value_counts(normalize=True) * 100).round(2)
print(f"count: {test_counts}, percent: {test_percent}")

# Baseline pipeline from PAN-CLEF

Best hyperparams for SVM from PAN-CLEF notebook

In [ ]:
pipeline = Pipeline([
    ("preprocessor", TweetPreprocessor(user_id_col=id_col, text_col=text_col)),

    ("features", FeatureUnion([
        ("tfidf_word",
         TfidfVectorizer(
             analyzer="word",
             ngram_range=(1,3),     # English
             min_df=2,
             max_df=1.0,
             sublinear_tf=True,
             lowercase=True,
             norm="l2"
         )
        ),

        ("tfidf_char",
         TfidfVectorizer(
             analyzer="char",
             ngram_range=(3,5),
             min_df=2,
             max_df=1.0,
             sublinear_tf=True,
             lowercase=True,
             norm="l2"
         )
        ),
    ])), # type: ignore

    ("svd",
     TruncatedSVD(
         n_components=300,
         random_state=int(os.getenv("RANDOM_SEED", 880055535))
     )
    ),

    ("clf",
     LinearSVC(
         C=1.0,
         random_state=int(os.getenv("RANDOM_SEED", 880055535)),
         class_weight="balanced"
     ),
    ),
])

# Training model on 100 tweets per author

In [ ]:
pipeline.fit(x_train, y_train)

# Importing 1 tweet per author model

Must have already trained model with one tweet per author

In [ ]:
one_tweet_model_path = os.path.join(MODELS_DIR, "model.joblib")

if os.path.exists(one_tweet_model_path):
    model = joblib.load(one_tweet_model_path)
else:
    raise FileNotFoundError(f"Model file not found: {one_tweet_model_path}")

# Comparison of 1 tweet vs 100 tweets per author models

## Test on 100 tweets per author dataset

In [ ]:
y_pred_100 = pipeline.predict(x_test)
y_pred_1 = model.predict(x_test)

labels = sorted(y_test.dropna().unique())

print("100 tweets per author model")
print(classification_report(y_test, y_pred_100, digits=4))

print("1 tweet per author model")
print(classification_report(y_test, y_pred_1, digits=4))

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred_100,
    labels=labels,
    cmap="Blues",
    ax=axes[0],
    colorbar=False,
)
axes[0].set_title("100 tweets per author")

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred_1,
    labels=labels,
    cmap="Blues",
    ax=axes[1],
    colorbar=False,
)
axes[1].set_title("1 tweet per author")

plt.tight_layout()
plt.show()

## Test on 1 tweet per author dataset

In [ ]:
test_df_1 = pd.read_csv(os.path.join(DATASETS_DIR, "one_tweet_dataset_test.csv"))

In [ ]:
x_test_1 = test_df_1["text"]
y_test_1 = test_df_1["gender_label"].map({'F': 0, 'M': 1})

In [ ]:
y_pred_100 = pipeline.predict(x_test_1)
y_pred_1 = model.predict(x_test_1)

labels = sorted(y_test_1.dropna().unique())

print("100 tweets per author model")
print(classification_report(y_test_1, y_pred_100, digits=4))

print("1 tweet per author model")
print(classification_report(y_test_1, y_pred_1, digits=4))

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ConfusionMatrixDisplay.from_predictions(
    y_test_1,
    y_pred_100,
    labels=labels,
    cmap="Blues",
    ax=axes[0],
    colorbar=False,
)
axes[0].set_title("100 tweets per author")

ConfusionMatrixDisplay.from_predictions(
    y_test_1,
    y_pred_1,
    labels=labels,
    cmap="Blues",
    ax=axes[1],
    colorbar=False,
)
axes[1].set_title("1 tweet per author")

plt.tight_layout()
plt.show()

# False positives and false negatives analysis

# Created dataset

In [ ]:
SAMPLE_SIZE = 5
RANDOM_STATE = os.getenv("RANDOM_SEED", 880055535)

def build_error_frames(texts, y_true, y_pred, model_name):
    df = pd.DataFrame({
        "row_id": pd.Series(range(len(y_true))),
        "text": pd.Series(texts).astype(str).reset_index(drop=True),
        "y_true": pd.Series(y_true).reset_index(drop=True),
        "y_pred": pd.Series(y_pred).reset_index(drop=True),
    })

    label_map = {0: "female", 1: "male"}
    df["true_label"] = df["y_true"].map(label_map).fillna(df["y_true"])
    df["pred_label"] = df["y_pred"].map(label_map).fillna(df["y_pred"])
    fp = df[(df["y_true"] == 0) & (df["y_pred"] == 1)].copy()
    fp["error_type"] = "false_positive"
    fp["model"] = model_name

    fn = df[(df["y_true"] == 1) & (df["y_pred"] == 0)].copy()
    fn["error_type"] = "false_negative"
    fn["model"] = model_name

    return fp, fn

def sample_errors(error_df, n=SAMPLE_SIZE, random_state=RANDOM_STATE):
    if error_df.empty:
        return error_df[["row_id", "text", "true_label", "pred_label", "model"]]
    out = error_df.sample(n=min(n, len(error_df)), random_state=random_state).copy()
    return out[["row_id", "text", "true_label", "pred_label", "model"]]

def inner_join_errors(df_a, df_b):
    if df_a.empty or df_b.empty:
        return pd.DataFrame(columns=["row_id", "text", "true_label", "pred_label_model", "pred_label_pipeline"])
    return df_a[["row_id", "text", "true_label", "pred_label"]].merge(
        df_b[["row_id", "text", "true_label", "pred_label"]],
        on=["row_id", "text", "true_label"],
        how="inner",
        suffixes=("_model", "_pipeline"),
    )

In [ ]:
created_pred_model = model.predict(x_test_1)
created_fp_model, created_fn_model = build_error_frames(x_test_1, y_test_1, created_pred_model, "model")
print(f"False Positive count: {len(created_fp_model)}")
print(f"False Negative count: {len(created_fn_model)}")
display(sample_errors(created_fp_model))
display(sample_errors(created_fn_model))

print("Created dataset: pipeline (PAN-CLEF baseline)")
created_pred_pipeline = pipeline.predict(x_test_1)
created_fp_pipeline, created_fn_pipeline = build_error_frames(x_test_1, y_test_1, created_pred_pipeline, "pipeline")
print(f"False Positive count: {len(created_fp_pipeline)}")
print(f"False Negative count: {len(created_fn_pipeline)}")
display(sample_errors(created_fp_pipeline))
display(sample_errors(created_fn_pipeline))

print("Created dataset: INNER JOIN errors (model ∩ pipeline)")
created_fp_inner = inner_join_errors(created_fp_model, created_fp_pipeline)
created_fn_inner = inner_join_errors(created_fn_model, created_fn_pipeline)
print(f"Inner join False Positive count: {len(created_fp_inner)}")
print(f"Inner join False Negative count: {len(created_fn_inner)}")
display(created_fp_inner.sample(n=min(SAMPLE_SIZE, len(created_fp_inner)), random_state=RANDOM_STATE) if len(created_fp_inner) > 0 else created_fp_inner)
display(created_fn_inner.sample(n=min(SAMPLE_SIZE, len(created_fn_inner)), random_state=RANDOM_STATE) if len(created_fn_inner) > 0 else created_fn_inner)

## PAN-CLEF dataset

In [ ]:
pan_pred_model = model.predict(x_test)
pan_fp_model, pan_fn_model = build_error_frames(x_test, y_test, pan_pred_model, "model")
print(f"False Positive count: {len(pan_fp_model)}")
print(f"False Negative count: {len(pan_fn_model)}")
display(sample_errors(pan_fp_model))
display(sample_errors(pan_fn_model))

print("PAN-CLEF dataset: pipeline (PAN-CLEF baseline)")
pan_pred_pipeline = pipeline.predict(x_test)
pan_fp_pipeline, pan_fn_pipeline = build_error_frames(x_test, y_test, pan_pred_pipeline, "pipeline")
print(f"False Positive count: {len(pan_fp_pipeline)}")
print(f"False Negative count: {len(pan_fn_pipeline)}")
display(sample_errors(pan_fp_pipeline))
display(sample_errors(pan_fn_pipeline))

print("PAN-CLEF dataset: INNER JOIN errors (model ∩ pipeline)")
pan_fp_inner = inner_join_errors(pan_fp_model, pan_fp_pipeline)
pan_fn_inner = inner_join_errors(pan_fn_model, pan_fn_pipeline)
print(f"Inner join False Positive count: {len(pan_fp_inner)}")
print(f"Inner join False Negative count: {len(pan_fn_inner)}")
display(pan_fp_inner.sample(n=min(SAMPLE_SIZE, len(pan_fp_inner)), random_state=RANDOM_STATE) if len(pan_fp_inner) > 0 else pan_fp_inner)
display(pan_fn_inner.sample(n=min(SAMPLE_SIZE, len(pan_fn_inner)), random_state=RANDOM_STATE) if len(pan_fn_inner) > 0 else pan_fn_inner)